# QC pass on cohort segmentation outliers

Stage 1 of the QC plan (visual adjudication of subjects whose DL segmentation falls outside the BraTS-2018 op-point band). This notebook is a thin shim over `hpgs.viz.qc`; all re-usable logic lives in `src/hpgs/viz/qc.py` so a future `scripts/qc_segmentation_outliers.py` (stage 2) can produce the same artefacts non-interactively.

**Inputs:**
- `data/derivatives_cohort50/sub-XXXX/metrics/sub-XXXX_metrics_dl_vs_gt.json` (PR-7d).
- `data/derivatives_cohort50/sub-XXXX/seg_dl/sub-XXXX_seg_brats3_dl.nii.gz` (PR-7d).
- `data/ucsf_pdgm_cohort50/sub-XXXX/sub-XXXX_T1c_bias.nii.gz` and `sub-XXXX_tumor_segmentation.nii.gz`.

**Strict-band thresholds (`hpgs.viz.qc.BRATS2018_OP_POINT`):** WT >= 0.85, TC >= 0.75, ET >= 0.70 Dice; HD95 <= 20 mm per compartment. A subject is flagged if any compartment falls below.

**Adjudication taxonomy:** `dl_miss` (DL genuinely mis-localises / misses substantial volume), `ref_mask_noise` (UCSF-PDGM reference is noisy; DL plausibly correct), `tiny_volume_hd95` (HD95 destabilised by tiny ET; boundary metric ill-defined), `image_artefact` (motion / metal / FOV in the input), `other` (document inline).

Outcome: a `keep_subjects` / `exclude_subjects` decision for the Table 2 / Table 3 rebuild.

## 1. Load metrics for all 50 subjects

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from hpgs.viz.qc import (
    BRATS2018_OP_POINT,
    HD95_MAX_MM_DEFAULT,
    classify_outliers,
    render_outlier_grid,
)

REPO = Path.cwd()
while not (REPO / "pyproject.toml").is_file() and REPO != REPO.parent:
    REPO = REPO.parent

COHORT_ROOT = REPO / "data" / "ucsf_pdgm_cohort50"
DERIV_ROOT = REPO / "data" / "derivatives_cohort50"

rows = classify_outliers(DERIV_ROOT)
print(f"Loaded {len(rows)} subjects from {DERIV_ROOT.relative_to(REPO)}")
print(f"Op-point band: {dict(BRATS2018_OP_POINT)}; HD95 cap: {HD95_MAX_MM_DEFAULT} mm")

In [ ]:
def rows_to_frame(rows):
    return pd.DataFrame(
        [
            {
                "subject": r.subject_id,
                "WT_dice": r.dice["WT"],
                "TC_dice": r.dice["TC"],
                "ET_dice": r.dice["ET"],
                "WT_hd95": r.hd95_mm["WT"],
                "TC_hd95": r.hd95_mm["TC"],
                "ET_hd95": r.hd95_mm["ET"],
                "flagged": r.flagged,
                "reasons": "; ".join(r.reasons),
            }
            for r in rows
        ]
    )

df = rows_to_frame(rows)
n_total = len(df)
n_flagged = int(df.flagged.sum())
print(f"{n_flagged} / {n_total} subjects flagged.")
df.style.format({c: "{:.3f}" for c in ["WT_dice", "TC_dice", "ET_dice"]}).format(
    {c: "{:.1f}" for c in ["WT_hd95", "TC_hd95", "ET_hd95"]}
).apply(lambda s: ["background-color: #fde68a" if v else "" for v in s], subset=["flagged"])

## 2. Cohort summary (sanity check vs. the segment-all log)

In [ ]:
summary = df[["WT_dice", "TC_dice", "ET_dice", "WT_hd95", "TC_hd95", "ET_hd95"]].agg(
    ["median", lambda s: s.quantile(0.25), lambda s: s.quantile(0.75), "min", "max"]
)
summary.index = ["median", "Q1", "Q3", "min", "max"]
summary.round(3)

## 3. Render flagged subjects (one row per subject: T1c | reference overlay | DL overlay)

In [ ]:
outliers = [r for r in rows if r.flagged]
print(f"Rendering {len(outliers)} outlier rows. WT yellow, TC orange, ET red.")
for r in outliers:
    print(
        f"  sub-{r.subject_id}  "
        f"WT {r.dice['WT']:.2f}/TC {r.dice['TC']:.2f}/ET {r.dice['ET']:.2f}  "
        f"HD95 WT {r.hd95_mm['WT']:.1f}/TC {r.hd95_mm['TC']:.1f}/ET {r.hd95_mm['ET']:.1f} mm  "
        f"|| {'; '.join(r.reasons)}"
    )

In [ ]:
fig = render_outlier_grid(
    outliers,
    cohort_root=COHORT_ROOT,
    deriv_root=DERIV_ROOT,
)
plt.show()

## 4. Adjudication

Adjudication taxonomy (one bullet per flagged subject):

    sub-XXXX -- <classification> -- <keep|exclude> -- <one-line rationale>

Classifications: `bundle_et_conservative`, `boundary_disagreement`, `dl_miss`, `ref_mask_noise`, `tiny_volume_hd95`, `image_artefact`, `other`.

**Cluster-mode check (Section 3 supplement, run on n=16 flagged):** every flagged reference mask is a complete BraTS-style annotation (NCR + ED + ET present, except sub-0388 where NCR=0 -- a microscopic difference). The 15 / 16 dominant pattern is the **MONAI Bundle being mildly conservative on the enhancing-tumor boundary** -- DL/REF ET-volume ratios cluster in the 0.55-0.90 range, with WT and TC ratios mostly 0.85-1.10. The single outlier from this pattern is **sub-0396** (DL/REF TC ratio 7.7, ET ratio 2.5) where the DL identifies a confluent necrotic-core + enhancing-rim within the central lesion that the reference labels predominantly as edema -- a substantive boundary disagreement that we cannot definitively adjudicate without expert review.

**Decision: keep all 50 subjects.** None of the flagged cases shows a hard segmentation failure (lobar miss, contralateral hallucination, image-artefact dominated mask) that would justify exclusion under a transparent, pre-registered rule. The bundle's ET-conservatism is a documented property of the model, not a per-subject defect; reporting medians/IQRs over the full n=50 cohort with `sub-0396` as a worst-case exemplar (Figure SX, `outputs/figures/fig_qc_sub0396_supplementary.pdf`) is the most honest framing.

**Per-subject adjudication:**

- sub-0018 -- bundle_et_conservative -- keep -- DL/REF ratios WT 0.93 / TC 0.93 / ET 0.82; representative of the dominant cluster
- sub-0026 -- bundle_et_conservative -- keep -- DL/REF ratios WT 0.95 / TC 0.92 / ET 0.83
- sub-0035 -- bundle_et_conservative -- keep -- DL under-predicts ET (ratio 0.54) on a large lesion; WT/TC still within band
- sub-0047 -- bundle_et_conservative -- keep -- ET ratio 1.00 but TC Dice 0.88 borderline due to NCR/ED redistribution
- sub-0071 -- bundle_et_conservative -- keep -- DL/REF ratios WT 0.85 / TC 0.86 / ET 0.70
- sub-0076 -- bundle_et_conservative -- keep -- ET ratio 0.52 (largest under-prediction in the cluster); WT 0.87, TC 0.87
- sub-0122 -- bundle_et_conservative -- keep -- small lesion (~5 cc TC); modest ratios
- sub-0155 -- bundle_et_conservative -- keep -- DL/REF ratios WT 1.06 / TC 1.05 / ET 0.77
- sub-0161 -- bundle_et_conservative -- keep -- DL/REF ratios WT 1.05 / TC 1.10 / ET 0.75
- sub-0210 -- bundle_et_conservative -- keep -- WT Dice 0.68 borderline (smallest WT in flagged set); ET ratio 0.65
- sub-0270 -- bundle_et_conservative -- keep -- DL/REF ratios WT 0.99 / TC 1.02 / ET 0.74
- sub-0388 -- bundle_et_conservative -- keep -- ref NCR=0 (microscopic); modest disagreement; small lesion
- sub-0389 -- bundle_et_conservative -- keep -- DL under-predicts both TC (0.63) and ET (0.57); WT 0.84
- sub-0392 -- bundle_et_conservative -- keep -- tiny core (~1.5 cc); within-cluster ratios; flagged on ET HD95 noise
- **sub-0396 -- boundary_disagreement -- keep -- exemplar; rendered as supplementary figure with footnote (see Methods)**
- sub-0505 -- bundle_et_conservative -- keep -- small lesion; ET ratio 0.76


## 5. Decision lists for the Table 2 / Table 3 rebuild

Locked-in per-subject decisions from section 4. **All 50 subjects retained** -- no exclusions. These two lists are the only inputs a future stage-2 producer (`scripts/qc_segmentation_outliers.py` + `--exclude-from-qc`) would need; with `exclude_subjects` empty, stage 2 is currently unnecessary and intentionally deferred.

In [ ]:
keep_subjects: list[str] = [
    "0018", "0026", "0035", "0047", "0071", "0076", "0122", "0155",
    "0161", "0210", "0270", "0388", "0389", "0392", "0396", "0505",
]

exclude_subjects: list[str] = []

print(f"keep    ({len(keep_subjects)}): {keep_subjects}")
print(f"exclude ({len(exclude_subjects)}): {exclude_subjects}")
overlap = set(keep_subjects) & set(exclude_subjects)
assert not overlap, f"keep/exclude overlap: {overlap}"
flagged_ids = {r.subject_id for r in outliers}
missing = flagged_ids - set(keep_subjects) - set(exclude_subjects)
if missing:
    print(f"WARNING: {len(missing)} flagged subjects not adjudicated yet: {sorted(missing)}")
else:
    print("All flagged subjects adjudicated; n=50 cohort retained for Table 2 / Table 3.")

## 6. Supplementary figure: sub-0396 worst-case exemplar

Render the standalone single-subject panel for sub-0396 and write the supplementary PDF that the Methods footnote points to.

In [ ]:
from hpgs.viz.qc import render_single_subject_panel

SUPP_OUT = REPO / "outputs" / "figures" / "fig_qc_sub0396_supplementary.pdf"
sub0396_row = next((r for r in rows if r.subject_id == "0396"), None)
assert sub0396_row is not None, "sub-0396 not in metrics table"

caption_0396 = (
    "Largest DL-vs-reference disagreement in the n=50 cohort (TC Dice 0.22, "
    "ET Dice 0.45). DL identifies a confluent necrotic core + enhancing rim "
    "(~17 cc) within the central lesion, while the UCSF-PDGM reference "
    "labels this region predominantly as edema (reference TC ~2 cc). "
    "Retained as a worst-case exemplar of cross-segmenter boundary "
    "disagreement; see Methods footnote for details."
)

result = render_single_subject_panel(
    "0396",
    cohort_root=COHORT_ROOT,
    deriv_root=DERIV_ROOT,
    out_path=SUPP_OUT,
    row=sub0396_row,
    caption=caption_0396,
)
print(f"wrote {result.relative_to(REPO)} ({result.stat().st_size:,} bytes)")

fig = render_single_subject_panel(
    "0396",
    cohort_root=COHORT_ROOT,
    deriv_root=DERIV_ROOT,
    row=sub0396_row,
    caption=caption_0396,
)
plt.show()